CNN Training

In [1]:
import os
from pathlib import Path
import pandas as pd
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights

In [2]:
if torch.cuda.is_available():
    device_name = "cuda"
elif torch.backends.mps.is_available():
    device_name = "mps"
else:
    device_name = "cpu"
    
print(f'Using device: {device_name}')
device = torch.device(device_name)

Using device: mps


Load Dataset

In [3]:
labels = pd.read_csv("../data/labels.csv")
image_paths = []
pattern_labels = []

for idx, row in labels.iterrows():
    folder = f"../data/frames/{row['division']}/{row['division']}_{row['id']}"
    for img_path in Path(folder).glob("*.jpg"):
        image_paths.append(str(img_path))
        pattern_labels.append(row['pattern'])


patterns = sorted(set(pattern_labels))
label2id = {p: i for i, p in enumerate(patterns)}

y = np.array([label2id[p] for p in pattern_labels])
len(image_paths)


119

Transform and Dataset class

In [4]:
tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class FrameDataset(Dataset):
    def __init__(self, paths, labels, tfm):
        self.paths = paths
        self.labels = labels
        self.tfm = tfm

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.tfm(img), self.labels[idx]

Train/Test split and DataLoaders

In [5]:
g = torch.Generator().manual_seed(42)
full_ds = FrameDataset(image_paths, y, tfm)
n = len(full_ds)
n_train = int(0.8 * n)
n_test = n - n_train


train_ds, test_ds = random_split(full_ds, [n_train, n_test], generator=g)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=32, shuffle=False)

n_train, n_test

(95, 24)

CNN model

In [6]:
num_classes = len(patterns)

model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/hp/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 99.1MB/s]


Training Loop

In [7]:
def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        if train:
            optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * yb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += yb.size(0)

    return total_loss / total, correct / total


Train for specified number of epochs

In [8]:
EPOCHS = 15
for epoch in range(1, EPOCHS+1):
    train_loss, train_acc = run_epoch(train_dl, train=True)
    test_loss, test_acc = run_epoch(test_dl, train=False)

    print(f"Epoch {epoch:02d} | "
          f"Train {train_loss:.4f}, Acc {train_acc:.3f} | "
          f"Val {test_loss:.4f}, Acc {test_acc:.3f}")

Epoch 01 | Train 0.4223, Acc 0.811 | Val 0.3151, Acc 0.833
Epoch 02 | Train 0.0467, Acc 1.000 | Val 0.1305, Acc 1.000
Epoch 03 | Train 0.0252, Acc 1.000 | Val 0.0424, Acc 1.000
Epoch 04 | Train 0.0063, Acc 1.000 | Val 0.0124, Acc 1.000
Epoch 05 | Train 0.0031, Acc 1.000 | Val 0.0042, Acc 1.000
Epoch 06 | Train 0.0027, Acc 1.000 | Val 0.0019, Acc 1.000
Epoch 07 | Train 0.0015, Acc 1.000 | Val 0.0012, Acc 1.000
Epoch 08 | Train 0.0010, Acc 1.000 | Val 0.0009, Acc 1.000
Epoch 09 | Train 0.0009, Acc 1.000 | Val 0.0008, Acc 1.000
Epoch 10 | Train 0.0007, Acc 1.000 | Val 0.0007, Acc 1.000
Epoch 11 | Train 0.0009, Acc 1.000 | Val 0.0006, Acc 1.000
Epoch 12 | Train 0.0014, Acc 1.000 | Val 0.0006, Acc 1.000
Epoch 13 | Train 0.0005, Acc 1.000 | Val 0.0006, Acc 1.000
Epoch 14 | Train 0.0010, Acc 1.000 | Val 0.0005, Acc 1.000
Epoch 15 | Train 0.0005, Acc 1.000 | Val 0.0005, Acc 1.000


Save model

In [9]:
Path("models").mkdir(exist_ok=True)

torch.save({
    "state_dict": model.state_dict(),
    "label2id": label2id,
    "id2label": {v: k for k, v in label2id.items()}
}, "models/cnn_sugarpush_sugartag.pth")

print("Saved model!")


Saved model!
